# DAIC-WOZ teacher training — Colab runner

Git is the source of truth. Training functions live under `scripts/`; this notebook only clones/pulls the repository and launches the selected script. Keep experiment outputs and checkpoints in Google Drive, not Git.


## 0. Clone or update Git

Add a Colab Secret named `GITHUB_TOKEN` with access to the private repository. The token is used only for Git authentication and is not printed or written into the repository.


In [ ]:
from pathlib import Path
import base64, subprocess, sys
from google.colab import userdata

GITHUB_REPO='engrkhubabahmad/reliability-aware-depression-kd'
REPO_DIR=Path('/content/reliability-aware-depression-kd')
GIT_REF='main'
token=userdata.get('GITHUB_TOKEN'); assert token, 'Add GITHUB_TOKEN in Colab Secrets.'
auth=base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git=['git','-c',f'http.extraHeader=AUTHORIZATION: basic {auth}']

if (REPO_DIR/'.git').is_dir():
    subprocess.run(git+['-C',str(REPO_DIR),'fetch','origin',GIT_REF],check=True)
else:
    subprocess.run(git+['clone',f'https://github.com/{GITHUB_REPO}.git',str(REPO_DIR)],check=True)
subprocess.run(['git','-C',str(REPO_DIR),'checkout',GIT_REF],check=True)
subprocess.run(git+['-C',str(REPO_DIR),'pull','--ff-only','origin',GIT_REF],check=True)
if str(REPO_DIR) not in sys.path: sys.path.insert(0,str(REPO_DIR))
print('Git ready:',REPO_DIR)


## 1. Install dependencies

Dependencies are versioned in Git. Existing compatible Colab packages are reused.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(REPO_DIR/'requirements-colab.txt')])


## 2. Frozen teachers — already completed

The frozen audio/text pipeline is now saved as `scripts/train_frozen_teachers.py`. Leave `RUN_FROZEN=False` for the current project because the frozen-teacher run has already been completed and cached. Set it to `True` only when intentionally rebuilding that stage. OOF is disabled.


In [ ]:
import runpy
RUN_FROZEN=False
if RUN_FROZEN:
    runpy.run_path(str(REPO_DIR/'scripts/train_frozen_teachers.py'),run_name='__main__')
else:
    print('Frozen stage skipped; using existing Drive results/cache.')


## 3. Full teacher fine-tuning — current stage

Runs `scripts/train_full_teachers_10epochs.py` directly from Git in this Colab runtime. It uses train/dev only, no OOF, saves epoch checkpoints in Drive, and resumes from `last.pt` when available.


In [ ]:
import runpy
runpy.run_path(str(REPO_DIR/'scripts/train_full_teachers_10epochs.py'),run_name='__main__')
